Build Query String

In [1]:
1+1

2

In [2]:
def build_query(parsed_output: dict) -> str:
    return parsed_output["normalized_address"]


External API Call (Notebook Cell)

In [54]:
import requests
import time

def fetch_geo_coordinates(parsed):
    query = ", ".join([
        # parsed["components"].get("building_name"),
        parsed["components"].get("area"),
        parsed["components"].get("city"),
        parsed["components"].get("state"),
        str(parsed["components"].get("pincode"))
    ])
    query = ", ".join([str(q) for q in query.split(", ") if q])

    url = "https://nominatim.openstreetmap.org/search"
    params = {
        "q": query,
        "format": "json",
        "limit": 1
    }
    headers = {
        "User-Agent": "AgenticGeoIntelligence/1.0 (contact: nlchiranth11@gmail.com)"
    }

    response = requests.get(url, params=params, headers=headers)
    time.sleep(1)
    print("Response code", response.status_code,  response.text)
    if response.status_code != 200:
        return None

    data = response.json()
    if not data:
        return None

    return {
        "latitude": float(data[0]["lat"]),
        "longitude": float(data[0]["lon"]),
        "confidence": 0.80,
        "source": "external"
    }


End-to-End Notebook Test

In [51]:
parsed_addressed = {
  "components": {
    "house_number": "Door Number 14",
    "building_name": None,
    "street": "5th cross",
    "landmark": None,
    "area": "N R Mohalla",
    "village": None,
    "taluk": None,
    "city": "Mysore",
    "district": None,
    "state": "Karnataka",
    "pincode": 570007
  },
  "normalized_address": "Door Number 14, 5th cross, N R Mohalla, Mysore, Karnataka, 570007",

}

In [5]:
cache_result = dict()
cache_result['decision'] = 'external_required'
cache_result

{'decision': 'external_required'}

In [55]:
parsed = parsed_addressed.copy()
# parsed["normalized_address"] = "Prestige Sunrise, Whitefield, Bengaluru, Karnataka, 560066"
# cache_result = check_cache(conn, parsed)#By passing
print("parsed  address ", parsed["normalized_address"])
if cache_result.get("decision") == "external_required":
    geo = fetch_geo_coordinates(parsed)

    if geo:
        print("EXTERNAL HIT", geo)
    else:
        print("EXTERNAL FAILED")


parsed  address  Door Number 14, 5th cross, N R Mohalla, Mysore, Karnataka, 570007
Response code 200 [{"place_id":233796615,"licence":"Data © OpenStreetMap contributors, ODbL 1.0. http://osm.org/copyright","osm_type":"way","osm_id":26652793,"lat":"12.3322420","lon":"76.6562780","class":"highway","type":"primary","place_rank":26,"importance":0.05339649576562139,"addresstype":"road","name":"T. N. Narasimhamurthy Circle","display_name":"T. N. Narasimhamurthy Circle, Narasimharaja Mohalla, Mysuru, Mysuru taluk, Mysuru District, Karnataka, 570001, India","boundingbox":["12.3319775","12.3324594","76.6560302","76.6565230"]}]
EXTERNAL HIT {'latitude': 12.332242, 'longitude': 76.656278, 'confidence': 0.8, 'source': 'external'}


Save External Result to DB

In [47]:
import sqlite3
import hashlib

conn = sqlite3.connect("geo_cache.db")
cursor = conn.cursor()

cursor.execute("""
CREATE TABLE IF NOT EXISTS geo_cache (
    address_hash TEXT PRIMARY KEY,
    normalized_address TEXT,
    city TEXT,
    state TEXT,
    pincode TEXT,
    latitude REAL,
    longitude REAL,
    confidence REAL,
    source TEXT,
    created_at TEXT DEFAULT CURRENT_TIMESTAMP
);
""")

conn.commit()

def generate_address_hash(components: dict) -> str:
    key_parts = [
        components.get("house_number"),
        components.get("building_name"),
        components.get("street"),
        components.get("landmark"),
        components.get("area"),
        components.get("village"),
        components.get("taluk"),
        components.get("city"),
        components.get("district"),
        components.get("state"),
        components.get("pincode")
    ]
    key = "|".join([str(p).lower() for p in key_parts if p])
    return hashlib.sha256(key.encode()).hexdigest()

def insert_cache(conn, record: dict):
    print("record ", record)
    cursor = conn.cursor()
    cursor.execute("""
        INSERT OR REPLACE INTO geo_cache
        VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, CURRENT_TIMESTAMP)
    """, (
        record["address_hash"],
        record["normalized_address"],
        record["city"],
        record["state"],
        record["pincode"],
        record["latitude"],
        record["longitude"],
        record["confidence"],
        record["source"]
    ))
    conn.commit()
    print('Successfully inserted into cache')

In [56]:
def save_external_result(conn, parsed, geo):
    record = {
        "address_hash": generate_address_hash(parsed["components"]),
        "normalized_address": parsed["normalized_address"],
        "city": parsed["components"]["city"],
        "state": parsed["components"]["state"],
        "pincode": parsed["components"]["pincode"],
        "latitude": geo["latitude"],
        "longitude": geo["longitude"],
        "confidence": geo["confidence"],
        "source": "external"
    }
    insert_cache(conn, record)

save_external_result(conn, parsed, geo)


record  {'address_hash': '707af3e16acaf7c810fd1780b91269c5fb595d93cc41308ab33f9f9c8b5798bc', 'normalized_address': 'Door Number 14, 5th cross, N R Mohalla, Mysore, Karnataka, 570007', 'city': 'Mysore', 'state': 'Karnataka', 'pincode': 570007, 'latitude': 12.332242, 'longitude': 76.656278, 'confidence': 0.8, 'source': 'external'}
Successfully inserted into cache
